Set True if you want to run a sweep

In [ ]:
do_sweep = False

Set systempath

In [ ]:
import sys
sys.path.append("../src")

Import everthing needed

In [ ]:
import pandas as pd
import numpy as np
import os
import wandb
import random
import torch
import torch.nn as nn
from torch.nn.utils import clip_grad_norm_, spectral_norm
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from transforms.feature_engineering import add_all_features, filter_business_hours, entries_per_day_per_site
from transforms.feature_engineering import (
    CONTINUOUS_FEATURE_COLUMNS,
    CATEGORICAL_FEATURE_COLUMNS,
    CYCLIC_FEATURE_COLUMNS,
    TARGET_COLUMN
)
from evaluation.comp_metrics import evaluate_all_metrics

Set wandb key (don't push to repository)

In [ ]:
os.environ["WANDB_API_KEY"] = "3aaf9f796df65417b3f5f8560b43875171b55805"

Set seed

In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

Load the data

In [ ]:
df_train = pd.read_csv("../data/classification/classification-train.csv")
df_test = pd.read_csv("../data/classification/classification-test.csv")

In [ ]:
print(f"Dataset shape: {df_train.shape}")
print(f"\nColumns: {df_train.columns.tolist()}")

In [ ]:
df_train = add_all_features(df_train)

In [ ]:
df_train = filter_business_hours(df_train)

In [ ]:
df_train.columns

In [ ]:
ENTRIES_PER_DAY = entries_per_day_per_site(df_train)

In [ ]:
print(ENTRIES_PER_DAY)

In [ ]:
def prepare_data():
    """Prepare and return all data splits"""
    df_train_site_a = df_train[0:19345]
    df_train_site_b = df_train[19345:38690]
    df_train_site_c = df_train[38690:58035]

    X_continuous_site_a = df_train_site_a[CONTINUOUS_FEATURE_COLUMNS].values
    X_continuous_site_b = df_train_site_b[CONTINUOUS_FEATURE_COLUMNS].values
    X_continuous_site_c = df_train_site_c[CONTINUOUS_FEATURE_COLUMNS].values

    X_categorical_site_a = df_train_site_a[CATEGORICAL_FEATURE_COLUMNS].values
    X_categorical_site_b = df_train_site_b[CATEGORICAL_FEATURE_COLUMNS].values
    X_categorical_site_c = df_train_site_c[CATEGORICAL_FEATURE_COLUMNS].values

    X_cyclic_site_a = df_train_site_a[CYCLIC_FEATURE_COLUMNS].values
    X_cyclic_site_b = df_train_site_b[CYCLIC_FEATURE_COLUMNS].values
    X_cyclic_site_c = df_train_site_c[CYCLIC_FEATURE_COLUMNS].values

    y_site_a = df_train_site_a[TARGET_COLUMN].values.reshape(-1, 1)
    y_site_b = df_train_site_b[TARGET_COLUMN].values.reshape(-1, 1)
    y_site_c = df_train_site_c[TARGET_COLUMN].values.reshape(-1, 1)

    y_site_a_unscaled = y_site_a.copy()

    scaler_X_site_a = StandardScaler()
    scaler_X_site_b = StandardScaler()
    scaler_X_site_c = StandardScaler()

    scaler_y_site_a = StandardScaler()
    scaler_y_site_b = StandardScaler()
    scaler_y_site_c = StandardScaler()

    X_scaled_site_a = scaler_X_site_a.fit_transform(X_continuous_site_a)
    X_scaled_site_b = scaler_X_site_b.fit_transform(X_continuous_site_b)
    X_scaled_site_c = scaler_X_site_c.fit_transform(X_continuous_site_c)

    y_site_a = scaler_y_site_a.fit_transform(y_site_a)
    y_site_b = scaler_y_site_b.fit_transform(y_site_b)
    y_site_c = scaler_y_site_c.fit_transform(y_site_c)

    X_site_a = np.concatenate([X_scaled_site_a, X_categorical_site_a, X_cyclic_site_a], axis=1).astype(np.float32)
    X_site_b = np.concatenate([X_scaled_site_b, X_categorical_site_b, X_cyclic_site_b], axis=1).astype(np.float32)
    X_site_c = np.concatenate([X_scaled_site_c, X_categorical_site_c, X_cyclic_site_c], axis=1).astype(np.float32)

    y_site_a = y_site_a.astype(np.float32)
    y_site_b = y_site_b.astype(np.float32)
    y_site_c = y_site_c.astype(np.float32)

    # Remove NaN entries
    mask_site_a = np.ones(len(X_site_a), dtype=bool)
    mask_site_b = np.ones(len(X_site_b), dtype=bool)
    mask_site_c = np.ones(len(X_site_c), dtype=bool)

    mask_site_a[0:ENTRIES_PER_DAY] = False
    mask_site_b[0:ENTRIES_PER_DAY] = False
    mask_site_c[0:ENTRIES_PER_DAY] = False

    X_site_a = X_site_a[mask_site_a]
    X_site_b = X_site_b[mask_site_b]
    X_site_c = X_site_c[mask_site_c]

    y_site_a = y_site_a[mask_site_a]
    y_site_b = y_site_b[mask_site_b]
    y_site_c = y_site_c[mask_site_c]

    return {
        'X_site_a': X_site_a, 'X_site_b': X_site_b, 'X_site_c': X_site_c,
        'y_site_a': y_site_a, 'y_site_b': y_site_b, 'y_site_c': y_site_c,
        'scaler_X_site_a': scaler_X_site_a, 'scaler_X_site_b': scaler_X_site_b, 'scaler_X_site_c': scaler_X_site_c,
        'scaler_y_site_a': scaler_y_site_a, 'scaler_y_site_b': scaler_y_site_b, 'scaler_y_site_c': scaler_y_site_c,
        'y_site_a_unscaled': y_site_a_unscaled
    }

In [ ]:
def create_sequences(X, y, seq_length):
    sequences_X = []
    sequences_y = []
    
    for i in range(len(X) - seq_length):
        sequences_X.append(X[i:i+seq_length])
        sequences_y.append(y[i+seq_length - 1])

    print(f"Sequences X: {len(sequences_X)}, Sequences Y: {len(sequences_y)}")
    
    return np.array(sequences_X), np.array(sequences_y)

In [ ]:
class PowerDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.FloatTensor(y)
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [ ]:
class Chomp1d(nn.Module):
    """Remove extra padding from causal conv to keep output length same as input"""
    def __init__(self, chomp_size):
        super().__init__()
        self.chomp_size = chomp_size

    def forward(self, x):
        return x[:, :, :-self.chomp_size].contiguous()


In [ ]:
class TemporalBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride, dilation, padding, dropout):
        super().__init__()
        # Replace weight_norm with spectral_norm
        self.conv1 = spectral_norm(nn.Conv1d(in_channels, out_channels, kernel_size,
                                              stride=stride, padding=padding, dilation=dilation))
        self.chomp1 = Chomp1d(padding)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(dropout)

        self.conv2 = spectral_norm(nn.Conv1d(out_channels, out_channels, kernel_size,
                                              stride=stride, padding=padding, dilation=dilation))
        self.chomp2 = Chomp1d(padding)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(dropout)

        self.net = nn.Sequential(self.conv1, self.chomp1, self.relu1, self.dropout1,
                                 self.conv2, self.chomp2, self.relu2, self.dropout2)
        self.downsample = nn.Conv1d(in_channels, out_channels, 1) if in_channels != out_channels else None
        self.relu = nn.ReLU()

    def forward(self, x):
        out = self.net(x)
        res = x if self.downsample is None else self.downsample(x)
        return self.relu(out + res)


In [ ]:
class TCN(nn.Module):
    def __init__(self, input_size, output_size, num_channels, kernel_size=2, dropout=0.2):
        super().__init__()
        layers = []
        num_levels = len(num_channels)
        for i in range(num_levels):
            in_ch = input_size if i == 0 else num_channels[i-1]
            out_ch = num_channels[i]
            layers.append(
                TemporalBlock(in_ch, out_ch, kernel_size, stride=1,
                              dilation=2**i, padding=(kernel_size-1)*(2**i), dropout=dropout)
            )
        self.network = nn.Sequential(*layers)
        self.fc = nn.Linear(num_channels[-1], output_size)

    def forward(self, x):
        # TCN expects (batch, channels, seq_len)
        x = x.transpose(1, 2)  # (B, seq_len, features) → (B, features, seq_len)
        y = self.network(x)
        y = y[:, :, -1]        # take last time step
        y = self.fc(y)
        return y

In [ ]:
def get_lr_with_warmup(epoch, base_lr, warmup_epochs):
    if warmup_epochs == 0 or epoch >= warmup_epochs:
        return base_lr
    else:
        return base_lr * (epoch + 1) / warmup_epochs

In [ ]:
def train_model(config=None):
    """
    Train the SimpleRNN model using a W&B sweep configuration.

    Includes:
    - Data preparation and scaling
    - Sequence creation
    - Model initialization
    - Training with custom false-positive penalty during non-DR hours
    - Validation, early stopping, and best-model saving
    - W&B metric logging
    """

    # -------------------------------
    # 1. Initialize Weights & Biases
    # -------------------------------
    wandb_run = wandb.init(
        project="AICOMP_Flextrack",
        entity="fabian-dubach-hochschule-luzern",
        config=config
    )
    config = wandb.config

    best_model_path = os.path.join(wandb_run.dir, "best_model.pt")

    print("\n" + "=" * 60)
    print("Starting run with config:")
    for k, v in dict(config).items():
        print(f"  {k}: {v}")
    print("=" * 60 + "\n")

    # -------------------------------
    # 2. Load + preprocess data
    # -------------------------------
    data = prepare_data()

    # Create sequences per site
    X_a, y_a = create_sequences(data['X_site_a'], data['y_site_a'], config.sequence_length)
    X_c, y_c = create_sequences(data['X_site_c'], data['y_site_c'], config.sequence_length)
    X_b, y_b = create_sequences(data['X_site_b'], data['y_site_b'], config.sequence_length)

    # Combine A + C for training
    X_train = np.vstack((X_a, X_c))
    y_train = np.vstack((y_a, y_c))

    X_val = X_b
    y_val = y_b

    # -------------------------------
    # 3. Build PyTorch Datasets
    # -------------------------------
    train_dataset = PowerDataset(X_train, y_train)
    val_dataset = PowerDataset(X_val, y_val)

    train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=False)
    val_loader = DataLoader(val_dataset, batch_size=config.batch_size, shuffle=False)

    # -------------------------------
    # 4. Extract metadata for metrics
    # -------------------------------
    # Inverse-transform building power for NMAE/NRMSE calculations
    train_bp_scaled = X_train[:, -1, 2]  # column index 2 = building power
    val_bp_scaled = X_val[:, -1, 2]

    # Helper array to inverse-transform only building power
    t_train = np.zeros((len(train_bp_scaled), 19))
    t_train[:, 2] = train_bp_scaled
    t_a = t_train[:len(X_a)]
    t_c = t_train[len(X_a):]

    bp_a = data['scaler_X_site_a'].inverse_transform(t_a)[:, 2]
    bp_c = data['scaler_X_site_c'].inverse_transform(t_c)[:, 2]
    train_building_power = np.concatenate([bp_a, bp_c])

    t_val = np.zeros((len(val_bp_scaled), 19))
    t_val[:, 2] = val_bp_scaled
    val_building_power = data['scaler_X_site_b'].inverse_transform(t_val)[:, 2]

    # Demand flag extraction
    train_demand_flags = np.argmax(X_train[:, -1, 30:33], axis=1) - 1
    val_demand_flags = np.argmax(X_val[:, -1, 30:33], axis=1) - 1

    # Site labels for metrics
    train_sites = np.concatenate([
        np.array(['Site A'] * len(bp_a)),
        np.array(['Site C'] * len(bp_c))
    ])
    val_sites = np.array(['Site B'] * len(X_val))

    # -------------------------------
    # 5. Initialize model + optimizer
    # -------------------------------
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # model = SimpleRNN(
    #     input_size=X_train.shape[2],
    #     hidden_size=config.hidden_size,
    #     num_layers=config.num_layers,
    #     output_size=1,
    #     dropout=config.dropout
    # ).to(device)

    tcn_channels = [64, 64, 128]  # customize number of channels per layer

    model = TCN(
        input_size=X_train.shape[2],
        output_size=1,
        num_channels=tcn_channels,
        kernel_size=3,
        dropout=config.dropout
    ).to(device)


    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=config.learning_rate,
        weight_decay=config.weight_decay
    )

    # -------------------------------
    # 6. Early stopping setup
    # -------------------------------
    early_stopping_patience = 10
    best_nmae = float('inf')
    epochs_without_improvement = 0

    print("Starting training...")

    # -------------------------------
    # 7. Training loop
    # -------------------------------
    for epoch in range(config.num_epochs):

        # Learning rate warmup
        current_lr = get_lr_with_warmup(epoch, config.learning_rate, config.warmup_epochs)
        for pg in optimizer.param_groups:
            pg['lr'] = current_lr

        model.train()
        train_loss = 0
        preds_all = []
        targets_all = []

        # ---------------------------
        # Training batches
        # ---------------------------
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)

            outputs = model(X_batch)

            # --- CUSTOM PENALTY ---
            base_loss = criterion(outputs, y_batch)

            # Identify "no DR" samples from one-hot flags
            dr_flags = X_batch[:, -1, 30:33]
            dr_ids = torch.argmax(dr_flags, dim=1)
            mask_no_dr = (dr_ids == 1)

            #FIXME
            if mask_no_dr.any():
                preds_no_dr = outputs[mask_no_dr].view(-1)
                # Clamp predictions before squaring to prevent explosion
                preds_no_dr_clamped = torch.clamp(preds_no_dr, -10, 10)
                penalty_term = torch.mean(preds_no_dr_clamped ** 2)
            else:
                penalty_term = torch.zeros(1, device=device)
            #FIXME

            penalty_weight = getattr(config, "fp_penalty_weight", 5.0)
            loss = base_loss + penalty_weight * penalty_term
            # ------------------------------------------------

            # Backprop
            optimizer.zero_grad()
            loss.backward()
            clip_grad_norm_(model.parameters(), config.gradient_clip_val)
            optimizer.step()

            #FIXME
            # Check for NaN in model parameters
            has_nan = False
            for name, param in model.named_parameters():
                if torch.isnan(param).any():
                    print(f"NaN detected in {name} at epoch {epoch}")
                    has_nan = True
                    break

            if has_nan:
                print("Training failed due to NaN. Stopping...")
                break

            # Also check predictions
            if torch.isnan(outputs).any():
                print(f"NaN in predictions at epoch {epoch}, batch {batch_idx}")
                continue  # skip this batch
            #FIXME

            train_loss += loss.item()
            preds_all.append(outputs.detach().cpu().numpy())
            targets_all.append(y_batch.cpu().numpy())

        train_loss /= len(train_loader)

        # -------------------------------
        # 8. Inverse transform predictions
        # -------------------------------
        preds_all = np.concatenate(preds_all)
        targets_all = np.concatenate(targets_all)

        
        preds_a = data['scaler_y_site_a'].inverse_transform(preds_all[:len(y_a)])
        preds_c = data['scaler_y_site_c'].inverse_transform(preds_all[len(y_a):])
        targs_a = data['scaler_y_site_a'].inverse_transform(targets_all[:len(y_a)])
        targs_c = data['scaler_y_site_c'].inverse_transform(targets_all[len(y_a):])

        train_preds = np.concatenate([preds_a, preds_c])
        train_targets = np.concatenate([targs_a, targs_c])

        # Training metrics
        train_metrics = evaluate_all_metrics(
            y_true=train_targets,
            y_pred=train_preds,
            site_labels=train_sites,
            building_power=train_building_power,
            demand_flags=train_demand_flags
        )

        # -------------------------------
        # 9. Validation
        # -------------------------------
        model.eval()
        val_loss = 0
        val_preds = []
        val_targs = []

        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                outputs = model(X_batch)

                loss = criterion(outputs, y_batch)
                val_loss += loss.item()

                val_preds.append(outputs.cpu().numpy())
                val_targs.append(y_batch.cpu().numpy())

        val_loss /= len(val_loader)

        val_preds = data['scaler_y_site_b'].inverse_transform(np.concatenate(val_preds))
        val_targs = data['scaler_y_site_b'].inverse_transform(np.concatenate(val_targs))

        val_metrics = evaluate_all_metrics(
            y_true=val_targs.flatten(),
            y_pred=val_preds.flatten(),
            site_labels=val_sites,
            building_power=val_building_power,
            demand_flags=val_demand_flags
        )

        # -------------------------------
        # 10. Log to W&B
        # -------------------------------
        wandb.log({
            "epoch": epoch,
            "learning_rate": current_lr,
            "train/loss": train_loss,
            "val/loss": val_loss,
            "train/nmae_mean": train_metrics['nmae_mean'],
            "val/nmae_mean": val_metrics['nmae_mean'],
            "train/base_loss": float(base_loss.item()),
            "train/penalty": float(penalty_term.item()),
            "train/total_loss": float(loss.item())
        })

        # Print occasionally
        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"\nEpoch {epoch+1}/{config.num_epochs}")
            print(f"LR: {current_lr:.6f}")
            print(f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
            print(f"Val NMAE(mean): {val_metrics['nmae_mean']:.2f}%")

        # -------------------------------
        # 11. Early stopping + save best
        # -------------------------------
        current_val_nmae = val_metrics["nmae_mean"]

        if current_val_nmae < best_nmae:
            best_nmae = current_val_nmae
            epochs_without_improvement = 0
            torch.save(model.state_dict(), best_model_path)
            print(f"Saved best model at epoch {epoch+1} (Val NMAE: {best_nmae:.2f}%)")
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= early_stopping_patience:
            print(f"\nEarly stopping at epoch {epoch+1}. Best Val NMAE: {best_nmae:.2f}%")
            break

    print(f"\nTraining completed! Best Val NMAE: {best_nmae:.2f}%")
    return best_nmae


In [ ]:
if do_sweep:
    sweep_config = {
        "method": "bayes",
        "metric": {"name": "val/nmae_mean", "goal": "minimize"},
        "parameters": {
            # Learning rate: wide, continuous log-uniform search
            "learning_rate": {"distribution": "log_uniform_values", "min": 1e-6, "max": 1e-2},

            # Batch sizes
            "batch_size": {"values": [32]},

            # Hidden layer sizes
            "hidden_size": {"values": [32, 64, 128, 256]},

            # Number of LSTM layers
            "num_layers": {"values": [1, 2, 3, 4]},

            # Dropout: continuous uniform
            "dropout": {"distribution": "uniform", "min": 0.0, "max": 0.6},

            # Weight decay: wide log-uniform search
            "weight_decay": {"distribution": "log_uniform_values", "min": 1e-6, "max": 1e-2},

            # Sequence lengths
            "sequence_length": {"values": [12, 24, 36, 48, 60]},

            # Training epochs
            "num_epochs": {"value": 50},
            "warmup_epochs": {"value": 5},

            # Gradient clipping
            "gradient_clip_val": {"distribution": "uniform", "min": 0.5, "max": 2.0},

            # FP penalty weight
            "fp_penalty_weight": {"values": [1.0, 10.0, 50.0, 100.0]},
        }
    }

    sweep_id = wandb.sweep(
        sweep_config,
        project="AICOMP_Flextrack",
        entity="fabian-dubach-hochschule-luzern"
    )
    print(f"Sweep created: {sweep_id}")

    wandb.agent(
        sweep_id,
        function=train_model,
        count=10
    )

else:
    default_config = {
        "hidden_size": 64,
        "num_layers": 2,
        "dropout": 0.2,
        "sequence_length": 24,
        "learning_rate": 1e-4,
        "weight_decay": 1e-4,
        "batch_size": 32,
        "num_epochs": 500,
        "warmup_epochs": 10,
        "gradient_clip_val": 1.0,
        "fp_penalty_weight": 10
    }

    train_model(default_config)